# Lesson 02 — Exercise Solutions

Worked answers to the five exercises at the end of
[`02_logistic_regression.ipynb`](02_logistic_regression.ipynb). Attempt them first. The
second one in particular is much more convincing when you have watched it fail yourself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from logistic_regression import (
    sigmoid, predict_proba, compute_cost, compute_gradient, gradient_descent,
    LogisticRegression, accuracy, confusion_matrix, precision_recall_f1,
)

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

# the two cluster dataset from the lesson
m = 200
X = np.vstack([
    rng.normal([-1.5, -1.0], 1.0, size=(m // 2, 2)),
    rng.normal([1.5, 1.0], 1.0, size=(m // 2, 2)),
])
y = np.concatenate([np.zeros(m // 2), np.ones(m // 2)])

w_fit, b_fit, _ = gradient_descent(X, y, np.zeros(2), 0.0, alpha=0.5, num_iters=3000)
print(f"refitted the lesson model: w = {w_fit.round(4)}, b = {b_fit:.4f}")

---
# Exercise 1 — Read the weights

> Confirm that increasing $x_1$ by one unit multiplies the odds by $e^{w_1}$. Compute the
> odds at two points that differ only in $x_1$ and check the ratio against $e^{w_1}$.

## Why it must be true

Start from the definition of odds and substitute the model:

$$\text{odds}(x) = \frac{p}{1 - p} = \frac{\sigma(z)}{1 - \sigma(z)} = e^{z} = e^{\,w \cdot x + b}$$

The middle step uses $1 - \sigma(z) = \sigma(-z)$, so the ratio collapses to a single
exponential. Now take two points that agree on every feature except $x_1$, where the
second sits one unit higher. Their logits differ by exactly $w_1$, so

$$\frac{\text{odds}(x + e_1)}{\text{odds}(x)} = \frac{e^{\,z + w_1}}{e^{\,z}} = e^{w_1}$$

The ratio does not depend on the starting point. This is what makes logistic regression
interpretable: each weight is a constant multiplicative effect on the odds, no matter
where in feature space you measure it. The quantity $e^{w_j}$ is called the **odds ratio**
for feature $j$, and it is what medical and social science papers report.

In [ ]:
def odds(X_points, w, b):
    p = predict_proba(X_points, w, b)
    return p / (1 - p)


# three base points, each paired with a copy one unit higher in x1 only
base = np.array([[0.0, 0.0], [-2.0, 1.5], [1.0, -3.0], [4.0, 3.0], [-4.0, -3.0]])
shifted = base + np.array([1.0, 0.0])

odds_base = odds(base, w_fit, b_fit)
odds_shifted = odds(shifted, w_fit, b_fit)

print(f"w1 = {w_fit[0]:.6f}, so the predicted odds ratio is exp(w1) = {np.exp(w_fit[0]):.6f}\n")
print(f"{'base point':>18}{'odds':>14}{'odds after +1 in x1':>22}{'ratio':>12}")
for point, o0, o1 in zip(base, odds_base, odds_shifted):
    print(f"{str(point):>18}{o0:>14.4g}{o1:>22.4g}{o1 / o0:>12.6f}")

print(f"\nevery ratio equals exp(w1) to machine precision: "
      f"{np.allclose(odds_shifted / odds_base, np.exp(w_fit[0]))}")

The odds themselves vary by orders of magnitude across the three base points, yet the
ratio is identical every time. Note also what is **not** constant: the change in
*probability*. Moving one unit in $x_1$ shifts the probability a lot near the decision
boundary and almost not at all out in the saturated tails, because the sigmoid is steep in
the middle and flat at the ends.

In [ ]:
prob_base = predict_proba(base, w_fit, b_fit)
prob_shifted = predict_proba(shifted, w_fit, b_fit)
print(f"{'base point':>18}{'P before':>12}{'P after':>12}{'change in P':>15}")
for point, p0, p1 in zip(base, prob_base, prob_shifted):
    print(f"{str(point):>18}{p0:>12.6f}{p1:>12.6f}{p1 - p0:>15.3g}")
print("\nThe effect on the odds is constant. The effect on the probability is not:")
print("near the boundary it is worth about 0.44, and out in the saturated tail")
print(f"at [4, 3] the very same one unit step is worth only {prob_shifted[3] - prob_base[3]:.2g}.")

---
# Exercise 2 — Make squared error fail on purpose

> Implement the squared error cost and its gradient for the logistic model, fit the 1D
> data from section 4 from several starting values, find a start where it stalls, and
> confirm cross-entropy reaches the same solution from every start.

## The implementation

$$J_{\text{MSE}}(w,b) = \frac{1}{2m}\sum_{i=1}^{m}\big(\sigma(z^{(i)}) - y^{(i)}\big)^2$$

The chain rule now keeps the $\sigma'(z)$ factor that cross-entropy cancels:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=1}^{m}\big(\sigma(z^{(i)}) - y^{(i)}\big)\,\sigma(z^{(i)})\big(1 - \sigma(z^{(i)})\big)\,x_j^{(i)}$$

That extra factor is the entire problem. It is near zero whenever the model is confident,
including when the model is confidently **wrong**.

In [ ]:
def cost_mse(X, y, w, b):
    return float(np.mean((predict_proba(X, w, b) - y) ** 2) / 2)


def gradient_mse(X, y, w, b):
    f = predict_proba(X, w, b)
    error = (f - y) * f * (1 - f)          # the sigmoid derivative survives here
    return (X.T @ error) / X.shape[0], float(error.sum() / X.shape[0])


def descend(X, y, w, b, alpha, num_iters, grad_fn, cost_fn):
    w, b, history = np.asarray(w, float).copy(), float(b), []
    for _ in range(num_iters):
        dj_dw, dj_db = grad_fn(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        history.append(cost_fn(X, y, w, b))
    return w, b, history


# gradient check before trusting any of it
X_1d = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]])
y_1d = np.array([0.0, 0.0, 0.0, 1.0, 1.0, 1.0])
w_c, b_c, eps = np.array([0.8]), -2.0, 1e-6
ana_w, ana_b = gradient_mse(X_1d, y_1d, w_c, b_c)
num_w = (cost_mse(X_1d, y_1d, w_c + eps, b_c) - cost_mse(X_1d, y_1d, w_c - eps, b_c)) / (2 * eps)
num_b = (cost_mse(X_1d, y_1d, w_c, b_c + eps) - cost_mse(X_1d, y_1d, w_c, b_c - eps)) / (2 * eps)
print(f"dJ/dw  analytic {ana_w[0]:12.8f}   numeric {num_w:12.8f}")
print(f"dJ/db  analytic {ana_b:12.8f}   numeric {num_b:12.8f}")

## Fitting from five different starting points

Same data, same learning rate, same number of iterations. Only the starting parameters
change.

In [ ]:
starts = [(0.0, 0.0), (-2.0, 6.0), (-5.0, 15.0), (-8.0, 25.0), (5.0, -20.0)]

# each metric group is 10 + 10 + 12 = 32 characters wide, so centre the group labels on 32
print(f"{'start (w, b)':>16} | {'squared error':^32} | {'cross-entropy':^32}")
print(f"{'':>16} | {'final w':>10}{'final b':>10}{'accuracy':>12} | {'final w':>10}{'final b':>10}{'accuracy':>12}")
print("-" * 86)
for w0, b0 in starts:
    w_m, b_m, _ = descend(X_1d, y_1d, np.array([w0]), b0, 1.0, 50_000, gradient_mse, cost_mse)
    w_x, b_x, _ = descend(X_1d, y_1d, np.array([w0]), b0, 1.0, 50_000, compute_gradient, compute_cost)
    acc_m = accuracy(y_1d, (predict_proba(X_1d, w_m, b_m) >= 0.5).astype(int))
    acc_x = accuracy(y_1d, (predict_proba(X_1d, w_x, b_x) >= 0.5).astype(int))
    print(f"{str((w0, b0)):>16} | {w_m[0]:>10.3f}{b_m:>10.3f}{acc_m:>12.3f}"
          f" | {w_x[0]:>10.3f}{b_x:>10.3f}{acc_x:>12.3f}")

**Squared error fails from three of the five starts**, landing at $50\%$ accuracy with a
*negative* weight. Cross-entropy converges to the same answer, $w \approx 11.5$ and
$b \approx -40.1$, from all five, differing only in the last few decimal places.

Look at what the failed runs actually learned. The weight has the wrong sign, so the model
believes larger $x$ means class 0. Every class 0 example is predicted correctly and every
class 1 example is predicted wrongly, which is where the $50\%$ comes from. The cost sits
at $0.25$, and that number is not arbitrary: three examples contribute a squared error of
about $1$ each and three contribute about $0$, giving a mean of $0.5$, halved by the
$\frac{1}{2}$ in the cost.

It is stuck because every prediction is saturated. The $\sigma(1 - \sigma)$ factor is
effectively zero for all six examples, so the gradient is effectively zero, so gradient
descent has nowhere to go. The model is on a **plateau**, not at a minimum, but for an
optimiser that only reads the gradient the difference is invisible.

## Mapping the basins of attraction

Rather than sampling five starting points, sweep a grid of them and record whether each
run ended up correct.

In [ ]:
w0_grid = np.linspace(-8, 8, 25)
b0_grid = np.linspace(-25, 25, 25)

outcome = {"squared error": np.zeros((len(b0_grid), len(w0_grid))),
           "cross-entropy": np.zeros((len(b0_grid), len(w0_grid)))}

for name, grad_fn, cost_fn in [("squared error", gradient_mse, cost_mse),
                               ("cross-entropy", compute_gradient, compute_cost)]:
    for i, b0 in enumerate(b0_grid):
        for j, w0 in enumerate(w0_grid):
            w_e, b_e, _ = descend(X_1d, y_1d, np.array([w0]), b0, 1.0, 2000, grad_fn, cost_fn)
            outcome[name][i, j] = accuracy(y_1d, (predict_proba(X_1d, w_e, b_e) >= 0.5).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, name in zip(axes, ["squared error", "cross-entropy"]):
    im = ax.pcolormesh(w0_grid, b0_grid, outcome[name], cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_xlabel("starting $w$"); ax.set_ylabel("starting $b$")
    ax.set_title(f"{name}: {np.mean(outcome[name] == 1.0):.0%} of starts reach 100% accuracy")
    plt.colorbar(im, ax=ax, label="final training accuracy")
plt.tight_layout(); plt.show()

Green means the run finished with every training example classified correctly, red means
it did not. Cross-entropy is green everywhere, because its cost is convex and has a single
minimum that gradient descent reaches from any starting point. Squared error has a large
red region, and which side of the divide you land on is decided entirely by where you
happened to initialise.

This is the practical meaning of "non-convex". It is not an abstract property. It means
your result depends on your random seed.

In [ ]:
# what the cost surface looks like, and why the red region traps the optimiser
w_surface = np.linspace(-8, 8, 200)
b_surface = np.linspace(-25, 25, 200)
WW, BB = np.meshgrid(w_surface, b_surface)
J_surface = np.array([[cost_mse(X_1d, y_1d, np.array([w]), b) for w in w_surface]
                      for b in b_surface])

fig, ax = plt.subplots(figsize=(7, 5))
filled = ax.contourf(WW, BB, J_surface, levels=25, cmap="viridis")
plt.colorbar(filled, ax=ax, label="$J_{MSE}$")
ax.contour(WW, BB, J_surface, levels=[0.05, 0.15, 0.24], colors="w", linewidths=0.8)
ax.set_xlabel("$w$"); ax.set_ylabel("$b$")
ax.set_title("Squared error surface: a wide flat plateau at $J = 0.25$ on the left")
plt.show()

plateau = (np.array([-6.0]), 2.0)
healthy = (np.array([1.0]), -3.5)
for name, (w_p, b_p) in [("on the plateau (w=-6, b=2)", plateau), ("near the answer (w=1, b=-3.5)", healthy)]:
    g_w, g_b = gradient_mse(X_1d, y_1d, w_p, b_p)
    print(f"{name:<32} cost {cost_mse(X_1d, y_1d, w_p, b_p):.6f}   "
          f"dJ/dw {g_w[0]:>11.3e}   dJ/db {g_b:>11.3e}")

g_w, _ = gradient_mse(X_1d, y_1d, *plateau)
print(f"\nOn the plateau the gradient is about {abs(g_w[0]):.0e}, so with alpha = 1.0 each step")
print("moves w by that much. It is also positive, and the update subtracts it, so w is")
print("driven further negative: the run slides deeper into the wrong region rather than")
print("escaping it. Both partial derivatives are dominated by the single example at x = 1,")
print("the only one that is not fully saturated.")

---
# Exercise 3 — Class imbalance

> Build a dataset where only $2\%$ of examples belong to class 1. Report accuracy,
> precision and recall, explain why accuracy looks excellent, then find the threshold that
> maximises $F_1$ and report again.

In [ ]:
n_imb = 2000
X_imb = rng.normal(0, 1, size=(n_imb, 2))
score = 1.5 * X_imb[:, 0] + X_imb[:, 1]
y_imb = (score > np.quantile(score, 0.98)).astype(float)   # top 2 percent are class 1

print(f"class counts: {int((y_imb == 0).sum())} negative, {int(y_imb.sum())} positive "
      f"({y_imb.mean():.1%} positive)")

model_imb = LogisticRegression(alpha=0.5, num_iters=4000).fit(X_imb, y_imb)
probs_imb = model_imb.predict_proba(X_imb)
pred_default = (probs_imb >= 0.5).astype(int)

cm = confusion_matrix(y_imb, pred_default)
precision, recall, f1 = precision_recall_f1(y_imb, pred_default)

print("\nconfusion matrix at the default threshold of 0.5")
print(f"                predicted 0   predicted 1")
print(f"  actually 0    {cm[0,0]:>11}   {cm[0,1]:>11}")
print(f"  actually 1    {cm[1,0]:>11}   {cm[1,1]:>11}")
print(f"\naccuracy  = {accuracy(y_imb, pred_default):.4f}")
print(f"precision = {precision:.4f}")
print(f"recall    = {recall:.4f}   <-- it is missing {cm[1,0]} of the {int(y_imb.sum())} positives")
print(f"F1        = {f1:.4f}")

trivial = np.zeros_like(y_imb, dtype=int)
print(f"\na model that always predicts 0 scores accuracy {accuracy(y_imb, trivial):.4f}")
print(f"so our model beats it by only {accuracy(y_imb, pred_default) - accuracy(y_imb, trivial):.4f}")

## Why accuracy looks excellent

Accuracy counts every correct prediction equally, and $98\%$ of the examples are negative.
A model that ignores the input entirely and always answers 0 already scores $98\%$. Our
model's accuracy is barely above that, so accuracy carries almost no information about
whether the model learned anything useful.

Recall is the number that exposes the problem. The model finds only $70\%$ of the
positives at the default threshold, missing 12 of the 40. Precision looks perfect, and that is not a
compliment here: the model achieves it by only committing to class 1 when it is
overwhelmingly confident, which is exactly the behaviour that loses recall.

**The cause is the threshold, not the model.** The default $0.5$ asks whether class 1 is
more likely than class 0. When positives are rare, the model has learned that they are
almost never more likely, so it rarely crosses $0.5$. The probabilities are still ranked
correctly, and lowering the threshold recovers the performance.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 197)
scores = np.array([precision_recall_f1(y_imb, (probs_imb >= t).astype(int)) for t in thresholds])
prec_curve, rec_curve, f1_curve = scores.T
best_t = thresholds[f1_curve.argmax()]

pred_best = (probs_imb >= best_t).astype(int)
p_b, r_b, f_b = precision_recall_f1(y_imb, pred_best)
cm_b = confusion_matrix(y_imb, pred_best)

print(f"best F1 threshold: {best_t:.3f}\n")
print(f"{'':<12}{'accuracy':>10}{'precision':>12}{'recall':>10}{'F1':>10}")
print(f"{'t = 0.5':<12}{accuracy(y_imb, pred_default):>10.4f}{precision:>12.4f}{recall:>10.4f}{f1:>10.4f}")
print(f"{f't = {best_t:.3f}':<12}{accuracy(y_imb, pred_best):>10.4f}{p_b:>12.4f}{r_b:>10.4f}{f_b:>10.4f}")
print(f"\nat the tuned threshold it catches {cm_b[1,1]} of {int(y_imb.sum())} positives "
      f"and raises {cm_b[0,1]} false alarms")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(thresholds, prec_curve, label="precision")
axes[0].plot(thresholds, rec_curve, label="recall")
axes[0].plot(thresholds, f1_curve, "k--", label="$F_1$")
axes[0].axvline(0.5, color="gray", ls=":", label="default")
axes[0].axvline(best_t, color="tab:green", ls="-.", label=f"best $F_1$ at {best_t:.2f}")
axes[0].set_xlabel("threshold"); axes[0].set_ylabel("score"); axes[0].legend(fontsize=8)
axes[0].set_title("Metrics against threshold")

axes[1].scatter(X_imb[y_imb == 0, 0], X_imb[y_imb == 0, 1], s=6, c="tab:blue", alpha=0.3, label="class 0")
axes[1].scatter(X_imb[y_imb == 1, 0], X_imb[y_imb == 1, 1], s=22, c="tab:red", marker="^", label="class 1")
x1_line = np.array([X_imb[:, 0].min(), X_imb[:, 0].max()])
for t, style, lbl in [(0.5, "k-", "boundary at t = 0.5"), (best_t, "g-.", f"boundary at t = {best_t:.2f}")]:
    shift = np.log(t / (1 - t))          # the boundary solves w.x + b = log(t / (1 - t))
    axes[1].plot(x1_line, (shift - model_imb.b - model_imb.w[0] * x1_line) / model_imb.w[1], style, label=lbl)
axes[1].set_xlabel("$x_1$"); axes[1].set_ylabel("$x_2$"); axes[1].legend(fontsize=8)
axes[1].set_title("Lowering the threshold moves the boundary outward")
plt.tight_layout(); plt.show()

Lowering the threshold to about $0.375$ raises recall from $0.70$ to $0.975$ while
precision stays at $1.0$, so it catches 39 of the 40 positives and still raises no false
alarms at all. The model was always capable of this. The default threshold was hiding it.

Worth noting for the right panel: thresholding at $t$ instead of $0.5$ moves the boundary
to where $w \cdot x + b = \log\frac{t}{1-t}$, which is a parallel shift of the same line.
Changing the threshold never rotates the boundary, it only slides it.

Other tools for imbalance, beyond tuning the threshold: weight the positive examples more
heavily in the cost, resample the training set, or report average precision instead of
accuracy. Tuning the threshold is the cheapest and should usually be tried first.

---
# Exercise 4 — Perfect separation

> Fit two clusters that a straight line separates with a wide gap. Print the norm of $w$
> every few hundred iterations. Explain why it keeps growing and why the cost approaches
> zero without reaching it.

In [ ]:
X_sep = np.vstack([rng.normal([-3.0, -3.0], 0.5, size=(50, 2)),
                   rng.normal([3.0, 3.0], 0.5, size=(50, 2))])
y_sep = np.concatenate([np.zeros(50), np.ones(50)])

w_s, b_s = np.zeros(2), 0.0
checkpoints, norms, costs = [], [], []
for block in range(60):
    w_s, b_s, _ = gradient_descent(X_sep, y_sep, w_s, b_s, alpha=0.5, num_iters=2000)
    checkpoints.append((block + 1) * 2000)
    norms.append(float(np.linalg.norm(w_s)))
    costs.append(compute_cost(X_sep, y_sep, w_s, b_s))

print(f"{'iteration':>12}{'||w||':>10}{'cost':>14}{'accuracy':>12}")
for k in (0, 4, 9, 24, 49, 59):
    acc = accuracy(y_sep, (predict_proba(X_sep, w_s, b_s) >= 0.5).astype(int))
    print(f"{checkpoints[k]:>12}{norms[k]:>10.4f}{costs[k]:>14.3e}{acc:>12.4f}")
print("\nthe cost is still falling and ||w|| is still climbing after 120,000 iterations")

## Why this happens

Once the data is perfectly separated, every example has $z^{(i)}$ on the correct side of
zero. Now scale the parameters up by a factor $c > 1$. Every logit becomes $c\,z^{(i)}$,
which pushes every prediction further towards the correct extreme, so every individual
loss gets smaller:

$$L = \log\big(1 + e^{z}\big) - yz \;\longrightarrow\; 0 \quad \text{as the correct logit grows}$$

So the cost can always be reduced further by simply making $w$ bigger. There is no finite
minimum. The optimum is at infinity, the cost decreases towards zero without ever reaching
it, and $\|w\|$ grows without bound. It grows slowly, roughly like the logarithm of the
iteration count, which is why this looks like convergence if you do not check.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(checkpoints, norms)
axes[0].set_xscale("log"); axes[0].set_xlabel("iteration"); axes[0].set_ylabel("$\\|w\\|$")
axes[0].set_title("The weight norm grows without settling")

axes[1].plot(checkpoints, costs)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("$J$")
axes[1].set_title("The cost falls towards zero, never reaching it")

# scaling the fitted parameters up always lowers the cost further
scales = np.linspace(0.2, 6, 100)
axes[2].plot(scales, [compute_cost(X_sep, y_sep, s * w_s, s * b_s) for s in scales])
axes[2].set_yscale("log"); axes[2].set_xlabel("scale factor $c$ applied to $w$ and $b$")
axes[2].set_ylabel("$J(cw, cb)$")
axes[2].set_title("Scaling up the same boundary always lowers $J$")
plt.tight_layout(); plt.show()

print("the decision boundary is unchanged by scaling, only the confidence changes:")
for c in (1.0, 3.0, 10.0):
    p = predict_proba(X_sep[:1], c * w_s, c * b_s)[0]
    print(f"  c = {c:5.1f}: cost {compute_cost(X_sep, y_sep, c * w_s, c * b_s):.3e}, "
          f"P(y=1) for the first point {p:.6e}")

The third panel is the clearest statement of the problem. Multiplying $w$ and $b$ by any
$c > 1$ leaves the decision boundary in exactly the same place, since $w \cdot x + b = 0$
and $c(w \cdot x + b) = 0$ describe the same set of points. All that changes is
confidence. The optimiser has no reason to stop, so it keeps inflating the weights to buy
cost reductions that do not improve a single prediction.

Three practical consequences:

1. The weights are meaningless in magnitude, so the odds ratios from exercise 1 become
   nonsense.
2. The model is maximally overconfident, reporting probabilities like $10^{-9}$ that no
   sample of 100 points could justify.
3. It never converges, so any stopping criterion based on the cost still improving will
   run forever.

**Regularisation is the fix.** Adding $\frac{\lambda}{2m}\|w\|^2$ to the cost charges for
large weights, so the reduction from inflating $w$ is eventually outweighed by the
penalty, and a finite optimum exists again. That is lesson 03.

---
# Exercise 5 — Mini-batch training

> Adapt the `sgd` function from the lesson 01 solutions to logistic regression. Check how
> little needs to change. Compare learning curves for batch sizes 1, 16 and the full
> dataset.

## How much needs to change

Nothing at all inside the function. The lesson 01 version calls `compute_gradient(X[idx],
y[idx], w, b)`, and logistic regression provides a `compute_gradient` with the same
signature, the same argument shapes and the same return shapes. Only the import at the top
of the file differs.

This is the practical payoff of the result from section 6 of the lesson. Because both
gradients reduce to (prediction minus label) times feature, every optimiser you write is
reusable across both models. Swapping the loss does not mean rewriting the training loop.

In [ ]:
def sgd(X, y, w, b, alpha, num_epochs, batch_size=1, decay=None, seed=1):
    # copied unchanged from the lesson 01 solutions, now running on logistic gradients
    rng_local = np.random.default_rng(seed)
    w, b = np.asarray(w, float).copy(), float(b)
    n_rows = X.shape[0]
    history, work = [], []
    examples_seen = 0

    for epoch in range(num_epochs):
        order = rng_local.permutation(n_rows)              # reshuffle each epoch
        for start in range(0, n_rows, batch_size):
            idx = order[start:start + batch_size]           # row indices for this batch
            step = alpha if decay is None else alpha / (1 + epoch / decay)
            dj_dw, dj_db = compute_gradient(X[idx], y[idx], w, b)
            w = w - step * dj_dw
            b = b - step * dj_db
            examples_seen += len(idx)
            history.append(compute_cost(X, y, w, b))        # full cost, for plotting only
            work.append(examples_seen)
    return w, b, np.array(history), np.array(work)


runs = {}
for label, batch_size in [("batch size 1", 1), ("batch size 16", 16), ("batch size 200 (full)", 200)]:
    w_r, b_r, hist, work = sgd(X, y, np.zeros(2), 0.0, alpha=0.5, num_epochs=40, batch_size=batch_size)
    runs[label] = (w_r, b_r, hist, work)
    acc = accuracy(y, (predict_proba(X, w_r, b_r) >= 0.5).astype(int))
    print(f"{label:<24} w = {w_r.round(3)}  b = {b_r:7.3f}  cost = {hist[-1]:.4f}  accuracy = {acc:.3f}")

w_ref, b_ref, _ = gradient_descent(X, y, np.zeros(2), 0.0, 0.5, 20_000)
print(f"{'long batch run':<24} w = {w_ref.round(3)}  b = {b_ref:7.3f}  "
      f"cost = {compute_cost(X, y, w_ref, b_ref):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, (_, _, hist, work) in runs.items():
    axes[0].plot(work, hist, lw=0.8, label=label)
axes[0].set_xscale("log"); axes[0].set_xlabel("example gradients computed (equal compute)")
axes[0].set_ylabel("$J$"); axes[0].legend(fontsize=8)
axes[0].set_title("Per unit of work, small batches make progress first")

for label, (_, _, hist, _) in runs.items():
    axes[1].plot(hist[-400:], lw=0.8, label=label)
axes[1].set_xlabel("update (last 400)"); axes[1].set_ylabel("$J$"); axes[1].legend(fontsize=8)
axes[1].set_title("Late training: batch size 1 is visibly noisy")
plt.tight_layout(); plt.show()

The left panel plots the cost against work done rather than against update count, which is
the only fair comparison. One full batch update costs 200 example gradients, and buys one
step. Two hundred single example updates cost the same and buy two hundred steps, each in
a noisy but roughly correct direction. Small batches win early for exactly the reason they
did in lesson 01.

The right panel shows the price. Batch size 1 never settles, because each update follows
the gradient of a single example rather than the average. It orbits the optimum in a noise
ball whose radius is set by the learning rate. Batch size 16 averages away most of that
variance while still taking many steps per pass, which is why real training uses
mini-batches in the range of tens to hundreds rather than either extreme.

Fixes for the noise, both carried over from lesson 01: decay the learning rate, or raise
the batch size.

---
## Recap

| exercise | the transferable lesson |
|---|---|
| 1 | Each weight is a constant multiplicative effect on the **odds**, not on the probability. |
| 2 | Non-convex means your answer depends on your initialisation. Squared error on a sigmoid gets stuck on saturated plateaus. |
| 3 | Under class imbalance, accuracy is uninformative and the default threshold of 0.5 is usually wrong. Tune it on the probabilities you already have. |
| 4 | Perfectly separable data has no finite optimum. The weights inflate forever and buy confidence rather than correctness. |
| 5 | The optimiser is independent of the loss. Because both gradients have the same form, the training loop is reusable without modification. |

Exercise 4 is the direct motivation for lesson 03, where a penalty on $\|w\|$ restores a
finite optimum and controls overfitting at the same time.